# 02 — Clean & Validate

Transform raw data into clean staging tables, then build analytics-ready mart tables.

**Input**: `data/raw/` (from notebook 01)  
**Output**: `data/staging/` + `data/mart/`

**Key business constant**: `COST_PER_LEAD = £55` (from actual account data)

Sections:
1. Load raw data
2. Clean insurance lead data → staging
3. Clean theLook data → staging
4. Data quality checks
5. Build mart tables (joined, analytics-ready)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
STAGING_DIR = PROJECT_ROOT / "data" / "staging"
MART_DIR = PROJECT_ROOT / "data" / "mart"

# Business constant — actual cost per lead from the account
COST_PER_LEAD = 55

## 1. Load Raw Data

In [2]:
# Insurance leads
leads_raw = pd.read_csv(RAW_DIR / "leads" / "insurance_leadgen_data.csv")
print(f"Leads: {leads_raw.shape}")

# theLook tables
users_raw = pd.read_parquet(RAW_DIR / "thelook" / "users.parquet")
orders_raw = pd.read_parquet(RAW_DIR / "thelook" / "orders.parquet")
order_items_raw = pd.read_parquet(RAW_DIR / "thelook" / "order_items.parquet")
events_raw = pd.read_parquet(RAW_DIR / "thelook" / "events_user_summary.parquet")
products_raw = pd.read_parquet(RAW_DIR / "thelook" / "products.parquet")
print(f"Users: {users_raw.shape}, Orders: {orders_raw.shape}, Items: {order_items_raw.shape}")
print(f"Events (user summary): {events_raw.shape}, Products: {products_raw.shape}")

# Spend tables
lead_spend = pd.read_csv(RAW_DIR / "spend" / "lead_spend.csv", parse_dates=["week_start"])
thelook_spend = pd.read_csv(RAW_DIR / "spend" / "thelook_spend.csv", parse_dates=["date"])
print(f"Lead spend: {lead_spend.shape}, theLook spend: {thelook_spend.shape}")

Leads: (7878, 13)
Users: (100000, 16), Orders: (124903, 9), Items: (180962, 11)
Events (user summary): (80125, 8), Products: (29120, 9)
Lead spend: (1404, 8), theLook spend: (2924, 6)


---
## 2. Clean Insurance Lead Data

Steps:
- Standardize column names to snake_case
- Fix `Patrial Postcode` typo, normalize to uppercase
- Extract `pc_area` (geographic letter prefix) from partial postcode
- Standardize verification status (NaN → `not_verified`)
- Group keywords into brand/intent-aware themes
- Create binary flags: `converted`, `is_invalid`, `high_value`
- Insurance-appropriate age bands
- Generate synthetic dates for time-based analysis

In [3]:
leads = leads_raw.copy()

# -- Column names --
leads.columns = (
    leads.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
leads = leads.rename(columns={"patrial_postcode": "partial_postcode"})

print("Columns:", list(leads.columns))

Columns: ['lead_id', 'lead_status', 'premium', 'age', 'gender', 'smoker', 'current_insurance', 'device_type', 'keyword', 'match_type', 'partial_postcode', 'cover_for', 'verification_status']


In [4]:
# -- Standardize values --
leads["partial_postcode"] = leads["partial_postcode"].str.upper().str.strip()
leads["verification_status"] = leads["verification_status"].fillna("not_verified")
leads["gender"] = leads["gender"].str.strip()
leads["smoker"] = leads["smoker"].str.strip().str.lower()           # yes/no
leads["current_insurance"] = leads["current_insurance"].str.strip()  # no, yes private, yes company
leads["device_type"] = leads["device_type"].str.strip()
leads["match_type"] = leads["match_type"].str.strip()

# Extract postcode area (geographic letter prefix, e.g. "SW" from "SW1A")
leads["pc_area"] = leads["partial_postcode"].str.extract(r"^([A-Z]{1,2})")[0]

print("Verification status:", leads["verification_status"].value_counts().to_dict())
print("Smoker:", leads["smoker"].value_counts().to_dict())
print(f"PC areas: {leads['pc_area'].nunique()} unique areas")

Verification status: {'not_verified': 6236, 'verified': 1642}
Smoker: {'no': 7156, 'yes': 722}
PC areas: 127 unique areas


In [5]:
# -- Keyword grouping (brand/intent-aware) --
def categorize_keyword(kw):
    """Group keywords by brand and intent — separates branded terms from generic."""
    kw = str(kw).lower().strip()
    if "bupa" in kw:
        return "Brand: Bupa"
    elif any(b in kw for b in ["axa", "aviva", "saga", "vitality"]):
        return "Brand: Other"
    elif any(w in kw for w in ["pre-existing", "pre existing", "preexisting", "existing condition"]):
        return "Pre-existing Conditions"
    elif any(w in kw for w in ["price", "cost", "quote", "cheap", "affordable", "how much"]):
        return "Price / Quote Intent"
    elif any(w in kw for w in ["compare", "comparison", "best", "review", "vs"]):
        return "Comparison / Research"
    elif any(w in kw for w in ["private health", "private medical"]):
        return "Generic: Private Health"
    elif any(w in kw for w in ["health insurance", "medical insurance"]):
        return "Generic: Health Insurance"
    else:
        return "Other"

leads["keyword_group"] = leads["keyword"].apply(categorize_keyword)
print(leads["keyword_group"].value_counts())

keyword_group
Generic: Private Health      3510
Brand: Bupa                  1695
Generic: Health Insurance    1009
Pre-existing Conditions       485
Price / Quote Intent          467
Comparison / Research         433
Other                         184
Brand: Other                   95
Name: count, dtype: int64


In [6]:
# -- Target flags --
leads["converted"] = (leads["lead_status"] == "Sold").astype(int)
leads["is_invalid"] = (leads["lead_status"] == "Invalid").astype(int)

# High value = top 20% premium among converted leads
converted_leads = leads[leads["converted"] == 1]
premium_threshold = converted_leads["premium"].quantile(0.80)
leads["high_value"] = ((leads["converted"] == 1) & (leads["premium"] >= premium_threshold)).astype(int)

print(f"Conversion rate: {leads['converted'].mean():.2%}")
print(f"Invalid rate:    {leads['is_invalid'].mean():.2%}")
print(f"High-value threshold: £{premium_threshold:,.0f}")
print(f"High-value leads: {leads['high_value'].sum()} ({leads['high_value'].mean():.2%} of all)")

Conversion rate: 4.90%
Invalid rate:    4.34%
High-value threshold: £2,177
High-value leads: 78 (0.99% of all)


In [7]:
# -- Age bands (insurance-appropriate bins) --
leads["age_band"] = pd.cut(
    leads["age"],
    bins=[0, 25, 30, 35, 40, 45, 50, 55, 60, 65, 75, 120],
    labels=["Under 25", "25-29", "30-34", "35-39", "40-44", "45-49", "50-54", "55-59", "60-64", "65-74", "75+"]
)
print(leads["age_band"].value_counts().sort_index())

age_band
Under 25     546
25-29        634
30-34        665
35-39       1089
40-44       1024
45-49        776
50-54        712
55-59        639
60-64        551
65-74        740
75+          502
Name: count, dtype: int64


In [8]:
# -- Synthetic dates --
# Assign dates spread over 12 months, roughly uniform with some weekly seasonality
np.random.seed(42)
n = len(leads)
date_range = pd.date_range("2025-01-01", "2025-12-31", freq="D")

# Slight weekday bias (more leads Mon-Fri)
weights = np.where(date_range.weekday < 5, 1.3, 0.7)
weights = weights / weights.sum()

leads["created_date"] = np.random.choice(date_range, size=n, p=weights)
leads = leads.sort_values("created_date").reset_index(drop=True)

print(f"Date range: {leads['created_date'].min().date()} to {leads['created_date'].max().date()}")
print(f"Avg leads/day: {n / 365:.1f}")

Date range: 2025-01-01 to 2025-12-31
Avg leads/day: 21.6


In [9]:
# Final lead staging table
print(f"Leads staging: {leads.shape}")
print(leads.dtypes)
leads.head()

Leads staging: (7878, 20)
lead_id                        object
lead_status                    object
premium                       float64
age                             int64
gender                         object
smoker                         object
current_insurance              object
device_type                    object
keyword                        object
match_type                     object
partial_postcode               object
cover_for                      object
verification_status            object
pc_area                        object
keyword_group                  object
converted                       int64
is_invalid                      int64
high_value                      int64
age_band                     category
created_date           datetime64[ns]
dtype: object


,lead_id,lead_status,premium,age,gender,smoker,current_insurance,device_type,keyword,match_type,partial_postcode,cover_for,verification_status,pc_area,keyword_group,converted,is_invalid,high_value,age_band,created_date
0,EDC9D129,Contacted,0.0,59,Female,no,no,Smartphone,private medical insurance uk,Exact,PE31,Self + Partner,not_verified,PE,Generic: Private Health,0,0,0,55-59,2025-01-01
1,FD26755A,Contacted,0.0,46,Female,no,no,Desktop,bupa cost,Exact,CV22,Self,not_verified,CV,Brand: Bupa,0,0,0,45-49,2025-01-01
2,E5CA4886,Contacted,0.0,48,Female,no,no,Smartphone,private health insurance with pre existing con...,Exact,PE29,Self,not_verified,PE,Pre-existing Conditions,0,0,0,45-49,2025-01-01
3,9C9FBC99,No Contact,0.0,36,Male,no,yes private,Smartphone,private health care,Exact,ME16,Self + Family,not_verified,ME,Generic: Private Health,0,0,0,35-39,2025-01-01
4,A433E331,Contacted,0.0,69,Female,no,yes private,Desktop,private medical insurance,Phrase,SW20,Self,verified,SW,Generic: Private Health,0,0,0,65-74,2025-01-01


---
## 3. Clean theLook Data

Build a user-level summary by joining users → orders → order_items → events.

Steps:
- Aggregate orders and order_items per user
- Join with events summary and user demographics
- Create conversion and high-value flags

In [10]:
# -- Order-level aggregation per user --
# Only count completed orders as revenue
completed_items = order_items_raw[order_items_raw["status"] == "Complete"].copy()

user_orders = (
    orders_raw
    .groupby("user_id")
    .agg(
        total_orders=("order_id", "nunique"),
        completed_orders=("status", lambda x: (x == "Complete").sum()),
        cancelled_orders=("status", lambda x: (x == "Cancelled").sum()),
        returned_orders=("status", lambda x: (x == "Returned").sum()),
        first_order=("created_at", "min"),
        last_order=("created_at", "max"),
    )
    .reset_index()
)

# Revenue from completed items only
user_revenue = (
    completed_items
    .groupby("user_id")
    .agg(
        total_revenue=("sale_price", "sum"),
        total_items=("sale_price", "count"),
        avg_item_price=("sale_price", "mean"),
    )
    .reset_index()
)

print(f"User orders: {user_orders.shape[0]:,} users")
print(f"User revenue: {user_revenue.shape[0]:,} users with completed orders")

User orders: 80,124 users
User revenue: 27,586 users with completed orders


In [11]:
# -- Build user-level summary --
# Start with all users
thelook_users = users_raw[["id", "age", "gender", "country", "city", "traffic_source", "created_at"]].copy()
thelook_users = thelook_users.rename(columns={"id": "user_id", "created_at": "user_created_at"})

# Join orders
thelook_users = thelook_users.merge(user_orders, on="user_id", how="left")
thelook_users = thelook_users.merge(user_revenue, on="user_id", how="left")

# Join events (drop the null user_id row from events)
events_clean = events_raw.dropna(subset=["user_id"]).copy()
thelook_users = thelook_users.merge(events_clean, on="user_id", how="left")

# Fill NaN for users with no orders/events
fill_zero_cols = [
    "total_orders", "completed_orders", "cancelled_orders", "returned_orders",
    "total_revenue", "total_items", "avg_item_price",
    "total_events", "session_count", "purchase_events", "cart_events", "product_views",
]
thelook_users[fill_zero_cols] = thelook_users[fill_zero_cols].fillna(0)

print(f"theLook user summary: {thelook_users.shape}")
thelook_users.head()

theLook user summary: (100000, 23)


,user_id,age,gender,country,city,traffic_source,user_created_at,total_orders,completed_orders,cancelled_orders,...,total_revenue,total_items,avg_item_price,total_events,session_count,purchase_events,cart_events,product_views,first_event,last_event
0,10479,63,M,Brasil,null,Search,2026-02-14 08:23:00+00:00,2.0,1.0,0.0,...,25.000000,1.0,25.000000,10,2,2,2,2,2026-02-14 13:10:54+00:00,2026-02-21 06:17:23+00:00
1,82675,56,F,Brasil,null,Search,2022-04-18 02:47:00+00:00,1.0,0.0,0.0,...,0.000000,0.0,0.000000,14,2,2,4,4,2025-04-23 06:33:39+00:00,2025-04-26 07:30:11+00:00
2,41297,49,F,Brasil,null,Search,2020-08-25 00:19:00+00:00,1.0,0.0,0.0,...,0.000000,0.0,0.000000,14,2,2,4,4,2025-04-21 04:34:23+00:00,2025-04-25 04:43:32+00:00
3,76783,68,F,Brasil,null,Organic,2021-09-16 07:34:00+00:00,2.0,0.0,0.0,...,0.000000,0.0,0.000000,10,2,2,2,2,2022-02-28 07:06:24+00:00,2022-03-10 21:07:01+00:00
4,60832,33,F,Brasil,null,Search,2023-02-23 14:19:00+00:00,2.0,1.0,1.0,...,37.950001,1.0,37.950001,19,3,3,5,5,2023-10-29 23:39:40+00:00,2024-03-08 02:57:07+00:00


In [12]:
# -- Target flags --
thelook_users["converted"] = (thelook_users["completed_orders"] > 0).astype(int)

# High value = top 20% revenue among converted users
converted_users = thelook_users[thelook_users["converted"] == 1]
revenue_threshold = converted_users["total_revenue"].quantile(0.80)
thelook_users["high_value"] = (
    (thelook_users["converted"] == 1) & (thelook_users["total_revenue"] >= revenue_threshold)
).astype(int)

# -- Derived features --
thelook_users["days_to_first_order"] = (
    (thelook_users["first_order"] - thelook_users["user_created_at"]).dt.total_seconds() / 86400
).round(1)

thelook_users["avg_order_value"] = (
    thelook_users["total_revenue"] / thelook_users["completed_orders"].replace(0, np.nan)
).round(2)

print(f"Conversion rate: {thelook_users['converted'].mean():.2%}")
print(f"High-value threshold: ${revenue_threshold:,.0f}")
print(f"High-value users: {thelook_users['high_value'].sum():,}")

Conversion rate: 27.59%
High-value threshold: $149
High-value users: 5,519


---
## 4. Data Quality Checks

In [13]:
def run_quality_checks(df, name, id_col, required_cols):
    """Run basic data quality checks and print results."""
    print(f"\n{'=' * 40}")
    print(f"Quality checks: {name}")
    print(f"{'=' * 40}")
    
    # Shape
    print(f"Rows: {df.shape[0]:,}, Cols: {df.shape[1]}")
    
    # Duplicates
    dupes = df[id_col].duplicated().sum()
    print(f"Duplicate {id_col}: {dupes} {'✓' if dupes == 0 else '✗'}")
    
    # Nulls in required columns
    for col in required_cols:
        if col in df.columns:
            nulls = df[col].isnull().sum()
            status = "✓" if nulls == 0 else f"✗ ({nulls} nulls)"
            print(f"  {col}: {status}")
    
    # Overall null summary
    total_nulls = df[required_cols].isnull().sum()
    if total_nulls.sum() == 0:
        print("No nulls in required columns ✓")
    
    return dupes == 0 and total_nulls.sum() == 0

In [14]:
# Lead quality checks
lead_ok = run_quality_checks(
    leads, "Insurance Leads", "lead_id",
    ["lead_id", "lead_status", "age", "gender", "device_type", "converted", "created_date"]
)

# Range checks
print(f"\nAge range: {leads['age'].min()} - {leads['age'].max()} ", end="")
print("✓" if 15 <= leads["age"].min() and leads["age"].max() <= 100 else "✗")
print(f"Negative premium: {(leads['premium'] < 0).sum()} ✓" if (leads["premium"] < 0).sum() == 0 else "✗")


Quality checks: Insurance Leads
Rows: 7,878, Cols: 20
Duplicate lead_id: 0 ✓
  lead_id: ✓
  lead_status: ✓
  age: ✓
  gender: ✓
  device_type: ✓
  converted: ✓
  created_date: ✓
No nulls in required columns ✓

Age range: 20 - 119 ✗
Negative premium: 0 ✓


In [15]:
# theLook quality checks
thelook_ok = run_quality_checks(
    thelook_users, "theLook Users", "user_id",
    ["user_id", "traffic_source", "converted", "total_revenue"]
)

# Range checks
print(f"\nNegative revenue: {(thelook_users['total_revenue'] < 0).sum()} ", end="")
print("✓" if (thelook_users["total_revenue"] < 0).sum() == 0 else "✗")
print(f"Users with orders: {(thelook_users['total_orders'] > 0).sum():,} / {len(thelook_users):,}")


Quality checks: theLook Users
Rows: 100,000, Cols: 27
Duplicate user_id: 0 ✓
  user_id: ✓
  traffic_source: ✓
  converted: ✓
  total_revenue: ✓
No nulls in required columns ✓

Negative revenue: 0 ✓
Users with orders: 80,124 / 100,000


---
## 5. Build Mart Tables

Create the final analytics-ready tables:
- **Lead mart**: leads joined to spend (for CAC/ROAS analysis)
- **theLook user mart**: user summary joined to spend
- **Campaign performance mart**: aggregated performance by date × source

In [16]:
# -- Lead mart --
# Aggregate leads by week × keyword_group × device × match_type to join with spend
leads["week_start"] = leads["created_date"].dt.to_period("W").apply(lambda x: x.start_time)

lead_weekly = (
    leads
    .groupby(["week_start", "keyword_group", "device_type", "match_type"])
    .agg(
        total_leads=("lead_id", "count"),
        conversions=("converted", "sum"),
        invalids=("is_invalid", "sum"),
        total_premium=("premium", "sum"),
        avg_premium=("premium", "mean"),
        high_value_leads=("high_value", "sum"),
    )
    .reset_index()
)

# Join with spend
lead_mart = lead_weekly.merge(
    lead_spend,
    on=["week_start", "keyword_group", "device_type", "match_type"],
    how="outer",
).fillna(0)

# Derived KPIs
lead_mart["conversion_rate"] = (lead_mart["conversions"] / lead_mart["total_leads"].replace(0, np.nan)).round(4)
lead_mart["invalid_rate"] = (lead_mart["invalids"] / lead_mart["total_leads"].replace(0, np.nan)).round(4)
lead_mart["cac"] = (lead_mart["spend"] / lead_mart["conversions"].replace(0, np.nan)).round(2)
lead_mart["roas"] = (lead_mart["total_premium"] / lead_mart["spend"].replace(0, np.nan)).round(2)
lead_mart["cost_per_lead"] = (lead_mart["spend"] / lead_mart["total_leads"].replace(0, np.nan)).round(2)

# Revenue-per-lead and net profit per lead (using COST_PER_LEAD constant)
lead_mart["rpl"] = (lead_mart["total_premium"] / lead_mart["total_leads"].replace(0, np.nan)).round(2)
lead_mart["net_per_lead"] = (lead_mart["rpl"] - COST_PER_LEAD).round(2)

print(f"Lead mart: {lead_mart.shape}")
print(f"Join coverage: {(lead_mart['total_leads'] > 0).mean():.1%} of rows have leads")
print(f"Overall RPL: £{lead_mart['total_premium'].sum() / lead_mart['total_leads'].sum():.2f}")
print(f"Overall Net/Lead: £{lead_mart['total_premium'].sum() / lead_mart['total_leads'].sum() - COST_PER_LEAD:.2f}")
lead_mart.head()

Lead mart: (2936, 21)
Join coverage: 52.2% of rows have leads
Overall RPL: £77.35
Overall Net/Lead: £22.35


,week_start,keyword_group,device_type,match_type,total_leads,conversions,invalids,total_premium,avg_premium,high_value_leads,...,clicks,spend,avg_cpc,conversion_rate,invalid_rate,cac,roas,cost_per_lead,rpl,net_per_lead
0,2024-12-30,Brand: Bupa,Desktop,Exact,6.0,2.0,0.0,2554.80,425.800000,0.0,...,0.0,0.0,0.0,0.3333,0.0,0.0,NaN,0.0,425.80,370.80
1,2024-12-30,Brand: Bupa,Smartphone,Exact,14.0,1.0,0.0,2712.96,193.782857,1.0,...,0.0,0.0,0.0,0.0714,0.0,0.0,NaN,0.0,193.78,138.78
2,2024-12-30,Comparison / Research,Desktop,Broad,3.0,0.0,0.0,0.00,0.000000,0.0,...,0.0,0.0,0.0,0.0000,0.0,NaN,NaN,0.0,0.00,-55.00
3,2024-12-30,Comparison / Research,Smartphone,Broad,1.0,0.0,0.0,0.00,0.000000,0.0,...,0.0,0.0,0.0,0.0000,0.0,NaN,NaN,0.0,0.00,-55.00
4,2024-12-30,Comparison / Research,Smartphone,Exact,1.0,0.0,1.0,0.00,0.000000,0.0,...,0.0,0.0,0.0,0.0000,1.0,NaN,NaN,0.0,0.00,-55.00


In [19]:
# -- theLook campaign performance mart --
# Aggregate users by month × traffic_source
thelook_users["user_month"] = thelook_users["user_created_at"].dt.to_period("M").apply(lambda x: x.start_time)

thelook_monthly = (
    thelook_users
    .groupby(["user_month", "traffic_source"])
    .agg(
        total_users=("user_id", "count"),
        conversions=("converted", "sum"),
        total_revenue=("total_revenue", "sum"),
        avg_revenue=("total_revenue", "mean"),
        high_value_users=("high_value", "sum"),
    )
    .reset_index()
)

# Aggregate spend to monthly
thelook_spend["month"] = thelook_spend["date"].dt.to_period("M").apply(lambda x: x.start_time)
spend_monthly = (
    thelook_spend
    .groupby(["month", "traffic_source"])
    .agg(spend=("spend", "sum"), clicks=("clicks", "sum"), impressions=("impressions", "sum"))
    .reset_index()
)

# Join
thelook_mart = thelook_monthly.merge(
    spend_monthly,
    left_on=["user_month", "traffic_source"],
    right_on=["month", "traffic_source"],
    how="outer",
)

# Use consistent date column (before fillna so types stay clean)
thelook_mart["month"] = thelook_mart["month"].combine_first(thelook_mart["user_month"])
thelook_mart = thelook_mart.drop(columns=["user_month"], errors="ignore")

# Fill numeric columns only
num_cols = thelook_mart.select_dtypes(include="number").columns
thelook_mart[num_cols] = thelook_mart[num_cols].fillna(0)

# Derived KPIs
thelook_mart["conversion_rate"] = (thelook_mart["conversions"] / thelook_mart["total_users"].replace(0, np.nan)).round(4)
thelook_mart["cac"] = (thelook_mart["spend"] / thelook_mart["conversions"].replace(0, np.nan)).round(2)
thelook_mart["roas"] = (thelook_mart["total_revenue"] / thelook_mart["spend"].replace(0, np.nan)).round(2)

print(f"theLook mart: {thelook_mart.shape}")
thelook_mart.head()

/var/folders/7k/007gkgf90hn56gqpkdw63_v40000gn/T/ipykernel_20689/4005566268.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  thelook_users["user_month"] = thelook_users["user_created_at"].dt.to_period("M").apply(lambda x: x.start_time)


theLook mart: (435, 13)


,traffic_source,total_users,conversions,total_revenue,avg_revenue,high_value_users,month,spend,clicks,impressions,conversion_rate,cac,roas
0,Display,46,12,1263.620005,27.470000,4,2019-01-01,0.0,0.0,0.0,0.2609,0.0,NaN
1,Email,59,13,1304.870011,22.116441,2,2019-01-01,0.0,0.0,0.0,0.2203,0.0,NaN
2,Facebook,83,24,1615.340011,19.461928,3,2019-01-01,0.0,0.0,0.0,0.2892,0.0,NaN
3,Organic,151,41,2916.389999,19.313841,5,2019-01-01,0.0,0.0,0.0,0.2715,0.0,NaN
4,Search,808,232,21532.480013,26.649109,48,2019-01-01,0.0,0.0,0.0,0.2871,0.0,NaN


---
## 6. Save All Staging & Mart Tables

In [20]:
# Staging (cleaned individual tables)
leads.to_parquet(STAGING_DIR / "leads_clean.parquet", index=False)
thelook_users.to_parquet(STAGING_DIR / "thelook_users_clean.parquet", index=False)

# Mart (joined analytics tables)
lead_mart.to_parquet(MART_DIR / "lead_performance.parquet", index=False)
thelook_mart.to_parquet(MART_DIR / "thelook_performance.parquet", index=False)

print("Saved to staging:")
print(f"  leads_clean.parquet          ({leads.shape[0]:,} rows)")
print(f"  thelook_users_clean.parquet   ({thelook_users.shape[0]:,} rows)")
print(f"\nSaved to mart:")
print(f"  lead_performance.parquet      ({lead_mart.shape[0]:,} rows)")
print(f"  thelook_performance.parquet   ({thelook_mart.shape[0]:,} rows)")

Saved to staging:
  leads_clean.parquet          (7,878 rows)
  thelook_users_clean.parquet   (100,000 rows)

Saved to mart:
  lead_performance.parquet      (2,936 rows)
  thelook_performance.parquet   (435 rows)
